# Model Architecture: Graph Neural Networks and Metric Learning

This notebook details the architecture of the Neural Thermodynamic Metric (NTM) model, including the GNN encoder and metric tensor learning.

## Table of Contents
1. [Overview](#1-overview)
2. [Molecular Graph Representation](#2-molecular-graph-representation)
3. [Graph Neural Network Encoder](#3-graph-neural-network-encoder)
4. [Metric Tensor Module](#4-metric-tensor-module)
5. [Full NTM Architecture](#5-full-ntm-architecture)
6. [Training Objective](#6-training-objective)

## 1. Overview

The Neural Thermodynamic Metric consists of three main components:

```
SMILES String → Molecular Graph → GNN Encoder → Embedding h ∈ ℝᵈ
                                                     ↓
                                              Metric Tensor M
                                                     ↓
Pair (A, B) → (h_A, h_B) → Riemannian Distance d_M(A,B) → Predicted Difficulty
```

### Design Principles

1. **End-to-end differentiable**: Learn representations and metric jointly
2. **Permutation invariant**: GNN respects molecular symmetry
3. **Geometrically meaningful**: Distance in embedding space has physical interpretation

## 2. Molecular Graph Representation

### Converting SMILES to Graphs

A molecule is represented as a graph $G = (V, E)$ where:
- **Nodes** $V$ = atoms
- **Edges** $E$ = bonds

### Node Features

For each atom, we compute:

| Feature | Description | Dimension |
|---------|-------------|----------|
| Atomic number | One-hot encoding | ~100 |
| Degree | Number of bonds | 1 |
| Formal charge | Ionic charge | 1 |
| Hybridization | sp, sp², sp³, etc. | ~5 |
| Aromaticity | Is aromatic ring member | 1 |
| Ring membership | In ring of size 3-8 | 6 |

### Edge Features

For each bond:

| Feature | Description | Dimension |
|---------|-------------|----------|
| Bond type | Single, double, triple, aromatic | 4 |
| Conjugated | Is conjugated | 1 |
| In ring | Is part of a ring | 1 |

In [ ]:
import torch
import numpy as np

try:
    from rdkit import Chem
    from rdkit.Chem import Draw, AllChem
    RDKIT_AVAILABLE = True
except ImportError:
    RDKIT_AVAILABLE = False
    print("RDKit not available. Install with: pip install rdkit")

def mol_to_graph(smiles):
    """
    Convert a SMILES string to a graph representation.
    
    Returns:
        dict with keys:
            - node_features: (n_atoms, n_node_features) tensor
            - edge_index: (2, n_edges) tensor
            - edge_features: (n_edges, n_edge_features) tensor
    """
    if not RDKIT_AVAILABLE:
        return None
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    # Node features
    node_features = []
    for atom in mol.GetAtoms():
        features = [
            atom.GetAtomicNum(),           # Atomic number
            atom.GetDegree(),              # Degree
            atom.GetFormalCharge(),        # Formal charge
            int(atom.GetHybridization()),  # Hybridization
            int(atom.GetIsAromatic()),     # Aromaticity
            atom.GetTotalNumHs(),          # Total H count
            int(atom.IsInRing()),          # In ring
        ]
        node_features.append(features)
    
    # Edge features and connectivity
    edge_index = []
    edge_features = []
    
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        
        # Add both directions (undirected graph)
        edge_index.extend([[i, j], [j, i]])
        
        bond_features = [
            float(bond.GetBondTypeAsDouble()),  # Bond order
            int(bond.GetIsConjugated()),        # Conjugated
            int(bond.IsInRing()),               # In ring
        ]
        edge_features.extend([bond_features, bond_features])
    
    return {
        'node_features': torch.tensor(node_features, dtype=torch.float32),
        'edge_index': torch.tensor(edge_index, dtype=torch.long).T,
        'edge_features': torch.tensor(edge_features, dtype=torch.float32),
        'batch': torch.zeros(len(node_features), dtype=torch.long)
    }

# Example
if RDKIT_AVAILABLE:
    smiles = "c1ccc(CC(=O)O)cc1"  # Phenylacetic acid
    graph = mol_to_graph(smiles)
    
    print(f"Molecule: {smiles}")
    print(f"Number of atoms: {graph['node_features'].shape[0]}")
    print(f"Number of bonds: {graph['edge_index'].shape[1] // 2}")
    print(f"Node feature dimension: {graph['node_features'].shape[1]}")
    print(f"Edge feature dimension: {graph['edge_features'].shape[1]}")
    
    # Display molecule
    mol = Chem.MolFromSmiles(smiles)
    Draw.MolToImage(mol, size=(300, 200))

## 3. Graph Neural Network Encoder

### Message Passing Framework

GNNs operate by iteratively updating node representations through message passing:

$$\mathbf{h}_i^{(l+1)} = \text{UPDATE}\left( \mathbf{h}_i^{(l)}, \text{AGGREGATE}\left( \{ \mathbf{m}_{j \to i} : j \in \mathcal{N}(i) \} \right) \right)$$

where:
- $\mathbf{h}_i^{(l)}$ is the hidden state of node $i$ at layer $l$
- $\mathcal{N}(i)$ is the neighborhood of node $i$
- $\mathbf{m}_{j \to i}$ is the message from node $j$ to node $i$

### Our Architecture: EdgeConv with Attention

We use a modified EdgeConv layer:

$$\mathbf{m}_{j \to i} = \text{MLP}\left( [\mathbf{h}_i \| \mathbf{h}_j - \mathbf{h}_i \| \mathbf{e}_{ij}] \right)$$

$$\mathbf{h}_i^{(l+1)} = \text{LayerNorm}\left( \mathbf{h}_i^{(l)} + \sum_{j \in \mathcal{N}(i)} \alpha_{ij} \mathbf{m}_{j \to i} \right)$$

where $\alpha_{ij}$ are learned attention weights and $\mathbf{e}_{ij}$ are edge features.

### Graph-Level Readout

After $L$ layers of message passing, we aggregate node features to get a graph-level embedding:

$$\mathbf{h}_G = \text{mean}\left( \{ \mathbf{h}_i^{(L)} : i \in V \} \right)$$

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_scatter import scatter_mean, scatter_add

class GraphConvLayer(nn.Module):
    """A single graph convolution layer with edge features."""
    
    def __init__(self, in_dim, out_dim, edge_dim):
        super().__init__()
        # Message MLP: processes [h_i || h_j - h_i || e_ij]
        self.message_mlp = nn.Sequential(
            nn.Linear(2 * in_dim + edge_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim)
        )
        
        # Attention for weighted aggregation
        self.attention = nn.Sequential(
            nn.Linear(out_dim, 1),
            nn.LeakyReLU(0.2)
        )
        
        # Update with residual connection
        self.update_mlp = nn.Linear(in_dim + out_dim, out_dim)
        self.norm = nn.LayerNorm(out_dim)
    
    def forward(self, x, edge_index, edge_attr):
        """
        Args:
            x: Node features (n_nodes, in_dim)
            edge_index: Edge connectivity (2, n_edges)
            edge_attr: Edge features (n_edges, edge_dim)
        
        Returns:
            Updated node features (n_nodes, out_dim)
        """
        src, dst = edge_index
        
        # Compute messages
        h_src = x[src]  # Source node features
        h_dst = x[dst]  # Destination node features
        
        # Message: [h_dst || h_src - h_dst || edge_features]
        messages = torch.cat([h_dst, h_src - h_dst, edge_attr], dim=-1)
        messages = self.message_mlp(messages)
        
        # Attention weights
        attn_scores = self.attention(messages)
        attn_weights = F.softmax(attn_scores, dim=0)  # Simplified; proper impl uses scatter_softmax
        
        # Aggregate messages (sum)
        aggregated = scatter_add(messages * attn_weights, dst, dim=0, dim_size=x.size(0))
        
        # Update with residual
        if x.size(-1) == aggregated.size(-1):
            out = self.norm(x + aggregated)
        else:
            out = self.norm(self.update_mlp(torch.cat([x, aggregated], dim=-1)))
        
        return out


class MolecularGNN(nn.Module):
    """Graph Neural Network for molecular embedding."""
    
    def __init__(self, node_dim=7, edge_dim=3, hidden_dim=64, num_layers=3):
        super().__init__()
        
        # Initial embedding
        self.node_embed = nn.Linear(node_dim, hidden_dim)
        self.edge_embed = nn.Linear(edge_dim, hidden_dim)
        
        # Graph convolution layers
        self.conv_layers = nn.ModuleList([
            GraphConvLayer(hidden_dim, hidden_dim, hidden_dim)
            for _ in range(num_layers)
        ])
        
        # Final projection
        self.output_proj = nn.Linear(hidden_dim, hidden_dim)
    
    def forward(self, node_features, edge_index, edge_features, batch):
        """
        Args:
            node_features: (n_nodes, node_dim)
            edge_index: (2, n_edges)
            edge_features: (n_edges, edge_dim)
            batch: (n_nodes,) - batch assignment for each node
        
        Returns:
            Graph-level embedding (batch_size, hidden_dim)
        """
        # Initial embeddings
        x = self.node_embed(node_features)
        edge_attr = self.edge_embed(edge_features)
        
        # Message passing
        for conv in self.conv_layers:
            x = conv(x, edge_index, edge_attr)
            x = F.relu(x)
        
        # Global mean pooling
        graph_embedding = scatter_mean(x, batch, dim=0)
        
        # Final projection
        return self.output_proj(graph_embedding)


# Test the GNN
print("MolecularGNN Architecture:")
gnn = MolecularGNN(node_dim=7, edge_dim=3, hidden_dim=64, num_layers=3)
print(gnn)
print(f"\nTotal parameters: {sum(p.numel() for p in gnn.parameters()):,}")

## 4. Metric Tensor Module

### Parameterization

We need to ensure the metric tensor $\mathbf{M}$ is **symmetric positive-definite** (SPD). Two approaches:

**Diagonal Metric** (simpler):
$$\mathbf{M} = \text{diag}(e^{m_1}, e^{m_2}, \ldots, e^{m_d})$$

Learning $m_i$ (log-weights) ensures positivity.

**Full Metric** (more expressive):
$$\mathbf{M} = \mathbf{L}\mathbf{L}^T + \epsilon \mathbf{I}$$

where $\mathbf{L}$ is lower-triangular (Cholesky factor) and $\epsilon$ ensures strict positivity.

### Interpretation of Metric Weights

- **Large diagonal entry** $M_{ii}$: Dimension $i$ is important for predicting difficulty
- **Off-diagonal entries** $M_{ij}$: Captures correlations between dimensions

In [ ]:
class MetricTensor(nn.Module):
    """
    Learnable Riemannian metric tensor.
    
    Supports diagonal (efficient) or full (expressive) parameterization.
    """
    
    def __init__(self, dim, metric_type='diagonal'):
        super().__init__()
        self.dim = dim
        self.metric_type = metric_type
        
        if metric_type == 'diagonal':
            # Learn log of diagonal elements
            self.log_weights = nn.Parameter(torch.zeros(dim))
        else:
            # Learn Cholesky factor
            self.L_diag = nn.Parameter(torch.ones(dim))  # Diagonal of L
            self.L_lower = nn.Parameter(torch.zeros(dim * (dim - 1) // 2))  # Lower triangular
    
    def get_weights(self):
        """Return diagonal weights (for interpretability)."""
        if self.metric_type == 'diagonal':
            return torch.exp(self.log_weights)
        else:
            M = self.get_full_metric()
            return torch.diag(M)
    
    def get_full_metric(self):
        """Return full metric tensor M."""
        if self.metric_type == 'diagonal':
            return torch.diag(torch.exp(self.log_weights))
        else:
            # Construct lower triangular L
            L = torch.zeros(self.dim, self.dim, device=self.L_diag.device)
            L[torch.eye(self.dim, dtype=bool)] = F.softplus(self.L_diag) + 0.1
            
            # Fill lower triangular part
            idx = torch.tril_indices(self.dim, self.dim, offset=-1)
            L[idx[0], idx[1]] = self.L_lower
            
            # M = L @ L^T
            return L @ L.T
    
    def compute_distance(self, h_a, h_b):
        """
        Compute Riemannian distance between embeddings.
        
        d_M(a, b) = sqrt((h_b - h_a)^T M (h_b - h_a))
        """
        diff = h_b - h_a
        
        if self.metric_type == 'diagonal':
            weights = torch.exp(self.log_weights)
            d_sq = torch.sum(diff ** 2 * weights, dim=-1)
        else:
            M = self.get_full_metric()
            d_sq = torch.sum(diff * (diff @ M), dim=-1)
        
        return torch.sqrt(d_sq + 1e-8)


# Example
dim = 64
metric = MetricTensor(dim, metric_type='diagonal')

# Simulate embeddings
h_a = torch.randn(1, dim)
h_b = torch.randn(1, dim)

euclidean = torch.norm(h_b - h_a)
riemannian = metric.compute_distance(h_a, h_b)

print(f"Embedding dimension: {dim}")
print(f"Euclidean distance: {euclidean.item():.4f}")
print(f"Riemannian distance: {riemannian.item():.4f}")
print(f"\nMetric weights (first 10): {metric.get_weights()[:10].detach().numpy()}")

## 5. Full NTM Architecture

### Putting It Together

The complete Neural Thermodynamic Metric model:

```
Input: (mol_A, mol_B) pair
    ↓
Graph Conversion: SMILES → Graph
    ↓
GNN Encoder: Graph → Embedding h ∈ ℝᵈ
    ↓
Metric Distance: d_M(h_A, h_B)
    ↓
Residual Path (optional): Adds direct embedding difference features
    ↓
Prediction Head: Maps to scalar difficulty prediction
```

### The Residual Path

We add a residual connection that allows the model to use raw embedding differences:

$$\hat{y} = \text{MLP}\left( [d_M(A, B) \| \mathbf{h}_B - \mathbf{h}_A \| \mathbf{h}_A + \mathbf{h}_B] \right)$$

This helps the model capture asymmetric effects (A→B vs B→A may have different difficulties).

In [ ]:
class NeuralThermodynamicMetric(nn.Module):
    """
    Neural Thermodynamic Metric for predicting RBFE difficulty.
    
    Architecture:
        1. GNN Encoder: Molecules → Embeddings
        2. Metric Tensor: Learned Riemannian metric
        3. Residual Path: Direct embedding features
        4. Prediction Head: Final difficulty prediction
    """
    
    def __init__(self, node_dim=7, edge_dim=3, hidden_dim=64, 
                 num_layers=3, metric_type='diagonal'):
        super().__init__()
        
        # GNN Encoder
        self.gnn = MolecularGNN(
            node_dim=node_dim,
            edge_dim=edge_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers
        )
        
        # Metric Tensor
        self.metric = MetricTensor(hidden_dim, metric_type=metric_type)
        
        # Residual path
        # Input: [riemannian_dist (1) | h_diff (d) | h_sum (d)]
        residual_dim = 1 + 2 * hidden_dim
        self.residual_net = nn.Sequential(
            nn.Linear(residual_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU()
        )
        
        # Prediction head
        self.pred_head = nn.Sequential(
            nn.Linear(hidden_dim // 2, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        
        self.hidden_dim = hidden_dim
        self.metric_type = metric_type
    
    def encode(self, graph):
        """Encode a molecular graph to embedding."""
        return self.gnn(
            graph['node_features'],
            graph['edge_index'],
            graph['edge_features'],
            graph['batch']
        )
    
    def compute_distance(self, h_a, h_b):
        """Compute Riemannian distance between embeddings."""
        return self.metric.compute_distance(h_a, h_b)
    
    def forward(self, graph_a, graph_b):
        """
        Predict difficulty of transformation A → B.
        
        Args:
            graph_a: Graph dict for molecule A
            graph_b: Graph dict for molecule B
        
        Returns:
            Predicted difficulty (batch_size,)
        """
        # Encode both molecules
        h_a = self.encode(graph_a)
        h_b = self.encode(graph_b)
        
        # Compute Riemannian distance
        riemannian_dist = self.compute_distance(h_a, h_b).unsqueeze(-1)
        
        # Residual features
        h_diff = h_b - h_a
        h_sum = h_a + h_b
        
        # Combine
        residual_input = torch.cat([riemannian_dist, h_diff, h_sum], dim=-1)
        residual_features = self.residual_net(residual_input)
        
        # Final prediction
        pred = self.pred_head(residual_features).squeeze(-1)
        
        return pred
    
    def get_metric_weights(self):
        """Return metric weights for interpretability."""
        return self.metric.get_weights()


# Test the full model
model = NeuralThermodynamicMetric(
    node_dim=7, edge_dim=3, hidden_dim=64, 
    num_layers=3, metric_type='diagonal'
)

print("Neural Thermodynamic Metric Model")
print("=" * 40)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Metric type: {model.metric_type}")
print(f"Hidden dimension: {model.hidden_dim}")

## 6. Training Objective

### Loss Function

We train to minimize the mean squared error between predicted and actual RBFE uncertainty:

$$\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} \left( \hat{y}_i - y_i \right)^2$$

where:
- $\hat{y}_i$ = predicted difficulty for pair $i$
- $y_i$ = ground truth (e.g., standard deviation of RBFE estimate from replicate runs)

### Regularization

To prevent metric collapse (all weights going to zero or infinity):

$$\mathcal{L}_{reg} = \lambda_1 \sum_i |\log m_i| + \lambda_2 \|\mathbf{M} - \mathbf{I}\|_F^2$$

This encourages metric weights to stay near 1 unless data strongly suggests otherwise.

### Training Loop

In [ ]:
def train_step(model, graph_a, graph_b, target, optimizer, reg_weight=0.01):
    """
    Single training step for the NTM model.
    
    Args:
        model: NeuralThermodynamicMetric model
        graph_a, graph_b: Molecular graphs for the pair
        target: Ground truth difficulty
        optimizer: Torch optimizer
        reg_weight: Metric regularization weight
    
    Returns:
        Loss value
    """
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    pred = model(graph_a, graph_b)
    
    # MSE loss
    mse_loss = F.mse_loss(pred, target)
    
    # Metric regularization (keep weights near 1)
    weights = model.get_metric_weights()
    reg_loss = reg_weight * torch.mean(torch.abs(torch.log(weights + 1e-8)))
    
    # Total loss
    loss = mse_loss + reg_loss
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    return loss.item(), mse_loss.item(), reg_loss.item()


def evaluate(model, dataloader):
    """
    Evaluate model on a dataset.
    
    Returns:
        dict with metrics (MSE, MAE, R², Kendall's τ)
    """
    model.eval()
    predictions = []
    targets = []
    
    with torch.no_grad():
        for graph_a, graph_b, target in dataloader:
            pred = model(graph_a, graph_b)
            predictions.append(pred)
            targets.append(target)
    
    predictions = torch.cat(predictions)
    targets = torch.cat(targets)
    
    mse = F.mse_loss(predictions, targets).item()
    mae = torch.mean(torch.abs(predictions - targets)).item()
    
    # R²
    ss_res = torch.sum((targets - predictions) ** 2)
    ss_tot = torch.sum((targets - targets.mean()) ** 2)
    r2 = 1 - (ss_res / ss_tot).item()
    
    return {'mse': mse, 'rmse': np.sqrt(mse), 'mae': mae, 'r2': r2}


print("Training utilities defined.")
print("\nKey components:")
print("  - train_step(): Single gradient update")
print("  - evaluate(): Compute metrics on validation set")

## Summary

### Model Architecture Recap

| Component | Purpose | Key Features |
|-----------|---------|-------------|
| **Graph Conversion** | SMILES → Graph | Node/edge features from RDKit |
| **GNN Encoder** | Graph → Embedding | Message passing, attention, pooling |
| **Metric Tensor** | Learned Riemannian metric | Diagonal or full parameterization |
| **Residual Path** | Capture asymmetric effects | Embedding differences |
| **Prediction Head** | Final scalar output | MLP |

### Key Design Decisions

1. **Diagonal vs Full Metric**: Diagonal is efficient but less expressive; full captures correlations
2. **Residual Path**: Helps model asymmetric transformations
3. **Regularization**: Prevents metric collapse

### Next Steps

In the next notebook, we'll explore:
- Computing geodesics through optimization
- Visualizing paths on the learned manifold
- Interpreting what the metric has learned